# Silver — ecommerce_rastreamento_entregas

Este notebook lê a camada Bronze de rastreamento, aplica as 10 regras de qualidade/negócio, salva a tabela Silver em Delta e registra o resumo das falhas em `squad1.dq_monitoring_logs`.

In [0]:

%run ../utils/utils

In [0]:
import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from functools import reduce
from datetime import datetime, timezone

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_rastreamento"
TABELA_DQ = "dq_monitoring_logs"

print(f"Iniciando processamento Silver - Rastreamento - Run ID: {RUN_ID}")

##  Anti-Join e Tabelas de Referência


In [0]:
# 1. Carrega Bronze de Rastreamento
try:
    df_bronze_rastreamento = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: Tabela Bronze de {TABELA_ALVO} não encontrada.")

# 2. Isola o Micro-lote (Apenas rastreios novos)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    df_micro_lote = df_bronze_rastreamento.join(df_silver_atual, "id_rastreamento", "left_anti")
else:
    df_micro_lote = df_bronze_rastreamento

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote de rastreamento: {qtd_novos}")

# 3. Referência de Pedidos (Regras 3, 7 e 9)
if delta_existe("bronze", "ecommerce_pedidos", STORAGE_OPTIONS):
    df_pedidos_ref = ler_delta("bronze", "ecommerce_pedidos", STORAGE_OPTIONS).select(
        F.col("id_pedido").alias("id_pedido_ref"),
        F.col("status_pedido").alias("status_pedido_ref")
    ).dropDuplicates(["id_pedido_ref"])
else:
    schema = StructType([
        StructField("id_pedido_ref", LongType(), True),
        StructField("status_pedido_ref", StringType(), True)
    ])
    df_pedidos_ref = spark.createDataFrame([], schema)

print("Tabela de referência de Pedidos carregada.")

## Aplicação das 10 Regras de Qualidade

In [0]:
if qtd_novos > 0:
    # Parâmetros
    status_validos = ['em separacao', 'coletado', 'em transito', 'saiu para entrega', 'entregue']
    regex_codigo = r"^[A-Z]{2}\d{9}$"
    
    # Janelas Analíticas (Window Functions)
    w_id_rastreamento = Window.partitionBy("id_rastreamento")
    w_pedido = Window.partitionBy("id_pedido_ecommerce")
    w_pedido_cronologia = Window.partitionBy("id_pedido_ecommerce").orderBy("dt_evento_ts")

    # Prepara os dados (Conversão de tipos, Joins e Janelas de agregação por pedido)
    df_base = df_micro_lote \
        .withColumn("dt_evento_ts", F.col("dt_evento").cast("timestamp")) \
        .join(df_pedidos_ref, df_micro_lote.id_pedido_ecommerce == df_pedidos_ref.id_pedido_ref, "left")

    df_base = df_base \
        .withColumn("qtd_id_rastreamento", F.count("*").over(w_id_rastreamento)) \
        .withColumn("dt_evento_anterior", F.lag("dt_evento_ts").over(w_pedido_cronologia)) \
        .withColumn("qtd_transportadoras", F.size(F.collect_set("id_transportadora").over(w_pedido))) \
        .withColumn("tem_evento_entregue", F.max(F.when(F.col("status_entrega") == 'entregue', 1).otherwise(0)).over(w_pedido)) \
        .withColumn("dt_coletado", F.max(F.when(F.col("status_entrega") == 'coletado', F.col("dt_evento_ts"))).over(w_pedido)) \
        .withColumn("dt_entregue", F.max(F.when(F.col("status_entrega") == 'entregue', F.col("dt_evento_ts"))).over(w_pedido))

    # Aplicação massiva das 10 regras
    df_silver_rastreamento = df_base \
        .withColumn("r1_id_rastreio_falhou", F.col("id_rastreamento").isNull() | (F.col("id_rastreamento").cast("string") == "") | (F.col("qtd_id_rastreamento") > 1)) \
        .withColumn("r2_status_invalido_falhou", F.col("status_entrega").isNull() | (~F.col("status_entrega").isin(status_validos))) \
        .withColumn("r3_pedido_fk_falhou", F.col("id_pedido_ecommerce").isNull() | F.col("id_pedido_ref").isNull()) \
        .withColumn("r4_dt_evento_futura_falhou", F.col("dt_evento_ts").isNull() | (F.col("dt_evento_ts") > F.current_timestamp())) \
        .withColumn("r5_codigo_padrao_falhou", F.col("codigo_rastreio").isNull() | (~F.col("codigo_rastreio").rlike(regex_codigo))) \
        .withColumn("r6_cronologia_invertida_falhou", F.col("dt_evento_anterior").isNotNull() & (F.col("dt_evento_ts") < F.col("dt_evento_anterior"))) \
        .withColumn("r7_entregue_sem_evento_falhou", (F.col("status_pedido_ref") == "Entregue") & (F.col("tem_evento_entregue") == 0)) \
        .withColumn("r8_sla_violado_falhou", F.col("dt_coletado").isNotNull() & F.col("dt_entregue").isNotNull() & (F.datediff(F.col("dt_entregue"), F.col("dt_coletado")) > 30)) \
        .withColumn("r9_evento_em_cancelado_falhou", F.col("status_pedido_ref") == "Cancelado") \
        .withColumn("r10_multiplas_transportadoras_falhou", (F.col("status_pedido_ref") != "Cancelado") & (F.col("qtd_transportadoras") > 1))

    print("Muralha de qualidade de rastreamento estruturada com sucesso.")
else:
    print("Etapa ignorada: não há micro-lote novo.")

## Catálogo de Regras e Logs

In [0]:
if qtd_novos > 0:
    catalogo_regras = [
        {"coluna": "r1_id_rastreio_falhou", "regra": "R1_ID_RASTREIO_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_status_invalido_falhou", "regra": "R2_STATUS_ENTREGA_INVALIDO", "severidade": "Critica"},
        {"coluna": "r3_pedido_fk_falhou", "regra": "R3_PEDIDO_FK_ORFAO", "severidade": "Critica"},
        {"coluna": "r4_dt_evento_futura_falhou", "regra": "R4_DATA_EVENTO_NULA_FUTURA", "severidade": "Critica"},
        {"coluna": "r5_codigo_padrao_falhou", "regra": "R5_CODIGO_RASTREIO_FORA_PADRAO", "severidade": "Critica"},
        {"coluna": "r6_cronologia_invertida_falhou", "regra": "R6_INVERSAO_CRONOLOGICA_EVENTOS", "severidade": "Critica"},
        {"coluna": "r7_entregue_sem_evento_falhou", "regra": "R7_PEDIDO_ENTREGUE_SEM_EVENTO_CORRESPONDENTE", "severidade": "Aviso"},
        {"coluna": "r8_sla_violado_falhou", "regra": "R8_SLA_ENTREGA_ACIMA_30_DIAS", "severidade": "Aviso"},
        {"coluna": "r9_evento_em_cancelado_falhou", "regra": "R9_MOVIMENTACAO_EM_PEDIDO_CANCELADO", "severidade": "Critica"},
        {"coluna": "r10_multiplas_transportadoras_falhou", "regra": "R10_MULTIPLAS_TRANSPORTADORAS_MESMO_PEDIDO", "severidade": "Aviso"}
    ]

    total_registros = df_silver_rastreamento.count()
    logs_list = []
    
    for r in catalogo_regras:
        qtd_falhas = df_silver_rastreamento.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID,
                TABELA_ALVO,
                r["regra"],
                "FAIL",
                r["severidade"],
                int(qtd_falhas),
                int(total_registros),
                datetime.now(timezone.utc),
                f"Bronze Delta ({TABELA_ALVO})"
            ))

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema=schema_dq_logs())
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema=schema_dq_logs())

    # Separação e redução usando o dict corretamente
    flags_criticas = [r["coluna"] for r in catalogo_regras if r["severidade"] == "Critica"]
    flags_avisos = [r["coluna"] for r in catalogo_regras if r["severidade"] == "Aviso"]

    condicao_invalida_critica = reduce(lambda a, b: a | b, [F.col(c) for c in flags_criticas])
    condicao_aviso = reduce(lambda a, b: a | b, [F.col(c) for c in flags_avisos]) if flags_avisos else F.lit(False)

    df_silver_rastreamento = df_silver_rastreamento \
        .withColumn("silver_linha_valida", ~condicao_invalida_critica) \
        .withColumn("silver_tem_aviso", condicao_aviso) \
        .withColumn("silver_processed_at", F.current_timestamp()) \
        .withColumn("silver_run_id", F.lit(RUN_ID))

    print("Logs processados:", df_dq_monitoring_logs_novos.count())
    display(df_dq_monitoring_logs_novos)
else:
    print("Etapa ignorada: não há micro-lote novo.")

## Gravação (Protegida e Particionada)

In [0]:
if qtd_novos > 0:
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id", "silver_tem_aviso"]
    
    df_silver_validos = df_silver_rastreamento \
        .filter(F.col("silver_linha_valida") == True) \
        .select(*colunas_finais)
        
    qtd_validos = df_silver_validos.count()
    print(f"Registros de Rastreamento aprovados para a Silver: {qtd_validos}")

    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos,
            camada="silver",
            tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS,
            mode="append",
            particionar=True # <-- IMPORTANTE: Rastreamento é gigante, então particionamos!
        )
        if sucesso_silver:
            print(f"Tabela Silver {TABELA_ALVO} atualizada com sucesso!")

    # Gravação dos Logs de Data Quality na RAIZ
    if df_dq_monitoring_logs_novos.count() == 0:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema=schema_dq_logs)

    if delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
        df_logs_historico = ler_delta(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS) \
            .filter(F.col("tabela") == TABELA_ALVO)
        
        condicao_join = [
            df_dq_monitoring_logs_novos.regra == df_logs_historico.regra,
            F.to_date(df_dq_monitoring_logs_novos.timestamp_execucao) == F.to_date(df_logs_historico.timestamp_execucao)
        ]
        
        df_logs_para_gravar = df_dq_monitoring_logs_novos.join(df_logs_historico, condicao_join, "left_anti")
    else:
        df_logs_para_gravar = df_dq_monitoring_logs_novos
        
    if df_logs_para_gravar.count() > 0 or not delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
        sucesso_logs = gravar_delta(
            df=df_logs_para_gravar,
            camada="", # Salva direto na raiz
            tabela=TABELA_DQ,
            storage_opts=STORAGE_OPTIONS,
            mode="append",
            particionar=False
        )
        if sucesso_logs:
            print("Logs de qualidade unificados processados na raiz com sucesso!")
else:
    print("Rotina finalizada sem alterações físicas.")

##  VALIDACAO


In [0]:
print("===== VALIDAÇÃO FINAL =====")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.orderBy(F.col("silver_processed_at").desc()).limit(20))
else:
    print(f"A tabela Silver {TABELA_ALVO} ainda não existe.")

if delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
    df_logs_validacao = ler_delta(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS)
    df_logs_filtrados = df_logs_validacao.filter(F.col("tabela") == TABELA_ALVO)
    
    print(f"Logs na {TABELA_DQ} para {TABELA_ALVO}:", df_logs_filtrados.count())
    display(df_logs_filtrados.orderBy(F.col("timestamp_execucao").desc()).limit(20))
else:
    print(f"Tabela {TABELA_DQ} ainda não existe no Data Lake.")